In [0]:
from pyspark.sql import functions as F

# 1. LIMPIEZA DE VENTAS (Reglas de Negocio)

# Leer la tabla desde Bronce en Unity Catalog
df_bronce_ventas = spark.table("databricks_proyecto_jhon.lakehouse.bronce_ventas")

df_plata_ventas = (df_bronce_ventas
    # Filtros de calidad (Eliminando la basura)
    .filter(F.col("estado") == "COMPLETADA")
    .filter(F.col("monto") > 0)
    .filter(F.col("fecha").isNotNull())
    .filter(F.col("id_cliente").isNotNull())
    
    # Asegurar que la fecha sea tipo Date (no solo texto)
    .withColumn("fecha", F.to_date(F.col("fecha")))
    
    # Sello de auditoría Plata
    .withColumn("_fecha_limpieza", F.current_timestamp())
    .drop("_fecha_carga") # Quitamos el sello viejo de bronce
)

# Guardar en Unity Catalog como Plata
(df_plata_ventas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("databricks_proyecto_jhon.lakehouse.plata_ventas")
)

print(f"✅ Ventas limpias en Plata: {df_plata_ventas.count()} registros válidos.")
display(df_plata_ventas)

In [0]:

# 2. TRANSFORMACIÓN BCRP (Aplanado y Formateo)

# Leer el único registro de Bronceo obtenido de la API
df_bronce_bcrp = spark.table("databricks_proyecto_jhon.lakehouse.bronce_tipo_cambio_bcrp")

# Usamos explode para distrubir la lista anidada periods y crear una fila por cada día
df_bcrp_explotado = df_bronce_bcrp.select(F.explode("periods").alias("dia_data"))

# Extraer los valores y filtrar los días sin datos ("n.d.")
df_bcrp_extraido = (df_bcrp_explotado
    .select(
        F.col("dia_data.name").alias("fecha_sucia"),          # Ej: "01.Ene.24" sin formato fecha
        F.col("dia_data.values")[0].alias("precio_dolar_str") # Ej: "values": ["3.732"] = "3.732"
    )
    .filter(F.col("precio_dolar_str") != "n.d.") # El BCRP no publica el tipo de cambio los fines de semana ni los feriados. Sin embargo, en su API, a veces incluyen esos días vacíos y en lugar de mandar un número, mandan el texto "n.d." (No Disponible).
)

# Traducir los meses en texto a números y armar la fecha real
df_plata_bcrp = (df_bcrp_extraido
    # Partir el texto "01.Ene.24" por los puntos
    .withColumn("dia", F.split(F.col("fecha_sucia"), "\\.")[0])
    .withColumn("mes_texto", F.split(F.col("fecha_sucia"), "\\.")[1])
    .withColumn("anio", F.concat(F.lit("20"), F.split(F.col("fecha_sucia"), "\\.")[2]))
    
    # Mapeo de meses en español (Incluyendo "Set" para el BCRP)
    .withColumn("mes", 
        F.when(F.col("mes_texto") == "Ene", "01")
         .when(F.col("mes_texto") == "Feb", "02")
         .when(F.col("mes_texto") == "Mar", "03")
         .when(F.col("mes_texto") == "Abr", "04")
         .when(F.col("mes_texto") == "May", "05")
         .when(F.col("mes_texto") == "Jun", "06")
         .when(F.col("mes_texto") == "Jul", "07")
         .when(F.col("mes_texto") == "Ago", "08")
         .when(F.col("mes_texto").isin("Sep", "Set"), "09")
         .when(F.col("mes_texto") == "Oct", "10")
         .when(F.col("mes_texto") == "Nov", "11")
         .when(F.col("mes_texto") == "Dic", "12")
    )
    
    # Construir la columna final de fecha y limpiar el tipo de cambio
    .withColumn("fecha", F.to_date(F.concat_ws("-", "anio", "mes", "dia")))
    .withColumn("tipo_cambio", F.col("precio_dolar_str").cast("double"))
    
    # Dejar solo las columnas limpias
    .select("fecha", "tipo_cambio")
    .withColumn("_fecha_limpieza", F.current_timestamp())
)

# Guardar en Unity Catalog como Plata
(df_plata_bcrp.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("databricks_proyecto_jhon.lakehouse.plata_tipo_cambio_bcrp")
)

print(f"✅ BCRP limpio en Plata: {df_plata_bcrp.count()} días registrados.")
display(df_plata_bcrp)